In [7]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# 2. Define File Paths
DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia-triplets-dataset.json")
SPLITS_PATH = os.path.join(_PROJECT_ROOT, "datasets", "splits_dbpedia.json")
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "final_model_mrr_0.4487.pt")

TOPKS = [3, 5, 10, 20]
BATCH = 32

# 3. Load the full dataset and the splits definition
raw_data = load_triplets(DATA_PATH)

with open(SPLITS_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

# 4. Filter for the TEST split using indices
# (Change "test_idx" to "test" if that is the actual key in your JSON)
test_indices = splits.get("test_idx", [])

# Since these are row numbers, we pull them directly from the raw_data list by their index
test_data = [raw_data[i] for i in test_indices]

print(f"Dataset Loaded -> Total: {len(raw_data)} | Test Split: {len(test_data)}")

# 5. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")

✓ Success! Project root set to: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings
Dataset Loaded -> Total: 4713 | Test Split: 472
Evaluation Ready -> Anchors: 472 | Corpus: 1881


In [8]:
# SPARQLCE
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen
evaluate_sentence_transformer('Qwen/Qwen2.5-0.5B-Instruct', 'Qwen2.5-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

Loading model from /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/sparqlce/models/final_model_mrr_0.4487.pt...

SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.2987  0.3533  0.2444  0.5177  0.6144    0.2987  0.4629
@5     0.2475  0.3974  0.2217  0.5305  0.6695    0.2475  0.4710
@10    0.1864  0.4698  0.1970  0.5403  0.7436    0.1864  0.4773
@20    0.1440  0.5555  0.1797  0.5445  0.8072    0.1440  0.4891

Baseline: TF-IDF
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1186  0.1884  0.1185  0.2373  0.2606    0.1186  0.2181
@5     0.0953  0.2087  0.1000  0.2499  0.3178    0.0953  0.2248
@10    0.0695  0.2374  0.0804  0.2582  0.3856    0.0695  0.2310
@20    0.0548  0.2841  0.0725  0.2645  0.4746    0.0548  0.2418



No sentence-transformers model found with name Qwen/Qwen2.5-0.5B-Instruct. Creating a new one with mean pooling.



Qwen2.5-0.5B
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1808  0.2077  0.1436  0.3372  0.3962    0.1808  0.2805
@5     0.1458  0.2311  0.1294  0.3471  0.4386    0.1458  0.2793
@10    0.1102  0.2684  0.1149  0.3564  0.5085    0.1102  0.2800
@20    0.0846  0.3340  0.1057  0.3620  0.5869    0.0846  0.2881
